# Compas Analysis (Python Version)

What follows are the calculations performed for ProPublica's analysis of the COMPAS Recidivism Risk Scores. It might be helpful to open [the methodology](https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm/) in another tab to understand the following.

This notebook is a faithful Python translation of the original R-based lecture notebook (`Lecture-01-alignment.ipynb`).

## Loading the Data

We select fields for severity of charge, number of priors, demographics, age, sex, compas scores, and whether each person was accused of a crime within two years.

## GenAI Disclosure Statement

In this course, generative artificial intelligence (GenAI) tools were used as learning aids to support understanding of Responsible ML concepts and workflow design. Specifically, OpenAI Codex/ChatGPT was used during brainstorming, preparatory research, and exploratory analysis.

This work is disclosed with integrity and transparency. All AI-assisted outputs were critically reviewed, validated, and revised by the student before submission. The submitted notebook reflects the student's own final intellectual work.


## SECTION 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## SECTION 2: Load Data

In [ ]:
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw_data = pd.read_csv(url)
print(len(raw_data))

In [ ]:
raw_data.head(10)

However not all of the rows are useable for the first round of analysis.

There are a number of reasons remove rows because of missing data:
* If the charge date of a defendants Compas scored crime was not within 30 days from when the person was arrested, we assume that because of data quality reasons, that we do not have the right offense.
* We coded the recidivist flag -- `is_recid` -- to be -1 if we could not find a compas case at all.
* In a similar vein, ordinary traffic offenses -- those with a `c_charge_degree` of 'O' -- will not result in Jail time are removed (only two of them).
* We filtered the underlying data from Broward county to include only those rows representing people who had either recidivated in two years, or had at least two years outside of a correctional facility.

## SECTION 3: Data Cleaning and Filtering

In [ ]:
# Select the same columns used in the R workflow
cols_to_keep = ["age", "c_charge_degree", "race", "age_cat",
                "score_text", "sex", "priors_count",
                "days_b_screening_arrest", "decile_score", "is_recid",
                "two_year_recid", "c_jail_in", "c_jail_out"]

df = raw_data[cols_to_keep].copy()

# ── Filtering (same conditions as R) ──
df = df[
    (df["days_b_screening_arrest"].between(-30, 30)) &
    (df["is_recid"] != -1) &
    (df["c_charge_degree"] != "O") &
    (df["score_text"] != "N/A")
].copy()

df.reset_index(drop=True, inplace=True)

# ── Type conversions ──
datetime_vars = ["c_jail_in", "c_jail_out"]
for col in datetime_vars:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Convert categorical columns to pandas Categorical type (equivalent to R factors)
factor_cols = ["c_charge_degree", "race", "age_cat", "score_text", "sex",
               "is_recid", "two_year_recid"]
for col in factor_cols:
    df[col] = df[col].astype(str).astype("category")

print(len(df))

## SECTION 4: Feature Engineering

Create derived factor variables matching the R workflow:
- `crime_factor` from `c_charge_degree`
- `age_factor` from `age_cat` with reference level "25 - 45"
- `race_factor` from `race` with reference level "Caucasian"
- `gender_factor` from `sex` with reference level "Male"
- `score_factor`: "LowScore" if `score_text == "Low"`, else "HighScore"

In [ ]:
df["crime_factor"] = pd.Categorical(df["c_charge_degree"])

df["age_factor"] = pd.Categorical(
    df["age_cat"],
    categories=["25 - 45", "Greater than 45", "Less than 25"]
)

df["race_factor"] = pd.Categorical(
    df["race"],
    categories=["Caucasian", "African-American", "Asian", "Hispanic",
                "Native American", "Other"]
)

# In R, the labels are applied as c("Female", "Male") and reference is "Male".
# The sex column already contains "Female" and "Male", so we just set the reference.
df["gender_factor"] = pd.Categorical(
    df["sex"],
    categories=["Male", "Female"]
)

# score_factor: "LowScore" if score_text == "Low", else "HighScore"
df["score_factor"] = pd.Categorical(
    np.where(df["score_text"] == "Low", "LowScore", "HighScore"),
    categories=["LowScore", "HighScore"]
)

In [ ]:
df.head(6)

In [ ]:
df.dtypes

Higher COMPAS scores are slightly correlated with a longer length of stay.

In [ ]:
df["length_of_stay"] = (df["c_jail_out"].dt.normalize() - df["c_jail_in"].dt.normalize()).dt.days
print(df["length_of_stay"].corr(df["decile_score"]))

## SECTION 5: EDA

After filtering we have the following demographic breakdown:

In [ ]:
# Equivalent to R: summary(df$age_cat)
print(df["age_cat"].value_counts().reindex(["25 - 45", "Greater than 45", "Less than 25"]))

In [ ]:
# Equivalent to R: summary(df$race)
print(df["race"].value_counts())

In [ ]:
n_total = len(df)
print("Black defendants: %.2f%%"      % (df["race"].value_counts()["African-American"] / n_total * 100))
print("White defendants: %.2f%%"      % (df["race"].value_counts()["Caucasian"]         / n_total * 100))
print("Hispanic defendants: %.2f%%"   % (df["race"].value_counts()["Hispanic"]          / n_total * 100))
print("Asian defendants: %.2f%%"      % (df["race"].value_counts()["Asian"]             / n_total * 100))
print("Native American defendants: %.2f%%" % (df["race"].value_counts()["Native American"] / n_total * 100))

In [ ]:
# Equivalent to R: summary(df$score_text)
print(df["score_text"].value_counts().reindex(["High", "Low", "Medium"]))

In [ ]:
# Equivalent to R: xtabs(~ sex + race, data=df)
print(pd.crosstab(df["sex"], df["race"]))

In [ ]:
# Equivalent to R: summary(df$sex)
print(df["sex"].value_counts())

In [ ]:
print("Men: %.2f%%"   % (df["sex"].value_counts()["Male"]   / n_total * 100))
print("Women: %.2f%%" % (df["sex"].value_counts()["Female"] / n_total * 100))

In [ ]:
# Number of recidivists
print(len(df[df["two_year_recid"] == "1"]))

In [ ]:
# Recidivism rate (%)
print(len(df[df["two_year_recid"] == "1"]) / len(df) * 100)

Judges are often presented with two sets of scores from the Compas system -- one that classifies people into High, Medium and Low risk, and a corresponding decile score. There is a clear downward trend in the decile scores as those scores increase for white defendants.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# Black defendants
black_scores = df[df["race"] == "African-American"]["decile_score"]
axes[0].hist(black_scores, bins=range(1, 12), align="left", rwidth=0.8, edgecolor="black")
axes[0].set_title("Black Defendant's Decile Scores")
axes[0].set_xlabel("Decile Score")
axes[0].set_xticks(range(1, 11))
axes[0].set_ylim(0, 650)

# White defendants
white_scores = df[df["race"] == "Caucasian"]["decile_score"]
axes[1].hist(white_scores, bins=range(1, 12), align="left", rwidth=0.8, edgecolor="black")
axes[1].set_title("White Defendant's Decile Scores")
axes[1].set_xlabel("Decile Score")
axes[1].set_xticks(range(1, 11))
axes[1].set_ylim(0, 650)

plt.tight_layout()
plt.show()

In [ ]:
# Equivalent to R: xtabs(~ decile_score + race, data=df)
print(pd.crosstab(df["decile_score"], df["race"]))

## Racial Bias in Compas

After filtering out bad rows, our first question is whether there is a significant difference in Compas scores between races. To do so we need to change some variables into factors, and run a logistic regression, comparing low scores to high scores.

## SECTION 6: Prepare Modeling Data

We create a binary target for the logistic regression: `score_factor` where "HighScore" = 1 and "LowScore" = 0.  
The `statsmodels` formula API uses `C(variable, Treatment(reference=...))` to set the reference level, which is equivalent to R's `relevel()`.

In [ ]:
# Binary target: 1 = HighScore, 0 = LowScore
df["score_binary"] = (df["score_factor"] == "HighScore").astype(int)

# Ensure two_year_recid is numeric for the model
df["two_year_recid_num"] = df["two_year_recid"].astype(str).astype(int)

## SECTION 7: Train Logistic Regression Model

This replicates the R call:  
```r
glm(score_factor ~ gender_factor + age_factor + race_factor +
    priors_count + crime_factor + two_year_recid,
    family = binomial(link = "logit"))
```

We use `statsmodels` formula interface with `C()` to set reference levels.

In [ ]:
formula = ("score_binary ~ C(gender_factor, Treatment(reference='Male'))"
           " + C(age_factor, Treatment(reference='25 - 45'))"
           " + C(race_factor, Treatment(reference='Caucasian'))"
           " + priors_count"
           " + C(crime_factor)"
           " + two_year_recid_num")

model_glm = smf.glm(formula=formula, data=df, family=sm.families.Binomial()).fit()
print(model_glm.summary())

**Note:** The coefficients should be very close to the R output. Minor floating-point differences are normal between R's `glm()` and Python's `statsmodels`.

Black defendants are 45% more likely than white defendants to receive a higher score correcting for the seriousness of their crime, previous arrests, and future criminal behavior.

In [ ]:
# Extract the intercept and the African-American coefficient from the fitted model
intercept = model_glm.params["Intercept"]
coef_black = model_glm.params["C(race_factor, Treatment(reference='Caucasian'))[T.African-American]"]

control = np.exp(intercept) / (1 + np.exp(intercept))
ratio_black = np.exp(coef_black) / (1 - control + (control * np.exp(coef_black)))
print(f"Black vs. Caucasian likelihood ratio: {ratio_black:.6f}")

Women are 19.4% more likely than men to get a higher score.

In [ ]:
coef_female = model_glm.params["C(gender_factor, Treatment(reference='Male'))[T.Female]"]
ratio_female = np.exp(coef_female) / (1 - control + (control * np.exp(coef_female)))
print(f"Female vs. Male likelihood ratio: {ratio_female:.6f}")

Most surprisingly, people under 25 are 2.5 times as likely to get a higher score as middle aged defendants.

In [ ]:
coef_young = model_glm.params["C(age_factor, Treatment(reference='25 - 45'))[T.Less than 25]"]
ratio_young = np.exp(coef_young) / (1 - control + (control * np.exp(coef_young)))
print(f"Under 25 vs. 25-45 likelihood ratio: {ratio_young:.6f}")

## SECTION 8: Generate Predicted Probabilities

In [ ]:
df["pred_prob"] = model_glm.predict(df)

## SECTION 9: Convert Probabilities to Binary Predictions Using 0.5 Threshold

In [ ]:
df["pred_class"] = np.where(df["pred_prob"] >= 0.5, "Recid", "No Recid")
df["pred_class"] = pd.Categorical(df["pred_class"], categories=["No Recid", "Recid"])

## SECTION 10: Overall Confusion Matrix and Evaluation Metrics

In [ ]:
print("\n Overall Confusion Matrix \n")

# Build confusion matrix matching R's layout: rows = Predicted, columns = Actual
overall_cm = pd.crosstab(
    df["pred_class"],
    df["two_year_recid"],
    rownames=["Predicted"],
    colnames=["Actual"]
)
print(overall_cm)

TP = overall_cm.loc["Recid",    "1"]
TN = overall_cm.loc["No Recid", "0"]
FP = overall_cm.loc["Recid",    "0"]
FN = overall_cm.loc["No Recid", "1"]
n = overall_cm.values.sum()

print(f"\nAccuracy : {(TP + TN) / n:.3f}")
print(f"Precision : {TP / (TP + FP):.3f}")
print(f"Recall : {TP / (TP + FN):.3f}")
print(f"FPR : {FP / (FP + TN):.3f}")
print(f"FNR : {FN / (FN + TP):.3f}")

## SECTION 11: Race-Level Fairness Metrics

This replicates the R grouped confusion-matrix analysis by race.

In [ ]:
print("\n\n Confusion Matrix by Race \n")

df["actual"] = df["two_year_recid"].astype(str).astype(int)
df["pred"]   = (df["pred_class"] == "Recid").astype(int)

race_metrics_list = []
for race_name, group in df.groupby("race", observed=False):
    n_group = len(group)
    tp = ((group["pred"] == 1) & (group["actual"] == 1)).sum()
    tn = ((group["pred"] == 0) & (group["actual"] == 0)).sum()
    fp = ((group["pred"] == 1) & (group["actual"] == 0)).sum()
    fn = ((group["pred"] == 0) & (group["actual"] == 1)).sum()

    accuracy  = round((tp + tn) / n_group, 3) if n_group > 0 else np.nan
    precision = round(tp / (tp + fp), 3) if (tp + fp) > 0 else np.nan
    recall    = round(tp / (tp + fn), 3) if (tp + fn) > 0 else np.nan
    fpr       = round(fp / (fp + tn), 3) if (fp + tn) > 0 else np.nan
    fnr       = round(fn / (fn + tp), 3) if (fn + tp) > 0 else np.nan

    race_metrics_list.append({
        "race": race_name,
        "n": n_group,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Accuracy": accuracy, "Precision": precision,
        "Recall": recall, "FPR": fpr, "FNR": fnr
    })

race_metrics = pd.DataFrame(race_metrics_list)
race_metrics = race_metrics.sort_values("n", ascending=False).reset_index(drop=True)
print(race_metrics.to_string(index=False))

## SECTION 12: Disparity Table Relative to Caucasian Baseline

In [ ]:
print("\n\n FPR and FNR Disparity by Race \n")

caucasian_fpr = race_metrics.loc[race_metrics["race"] == "Caucasian", "FPR"].values[0]
caucasian_fnr = race_metrics.loc[race_metrics["race"] == "Caucasian", "FNR"].values[0]

disparity = race_metrics[["race", "n", "FPR", "FNR"]].copy()
disparity["delta_FPR"] = round(disparity["FPR"] - caucasian_fpr, 3)
disparity["delta_FNR"] = round(disparity["FNR"] - caucasian_fnr, 3)

print(disparity.to_string(index=False))

## SECTION 13: Short Interpretation of Results

### Key Findings

1. **Racial disparity in COMPAS scores:** The logistic regression confirms that African-American defendants are approximately 45% more likely than Caucasian defendants to receive a higher COMPAS risk score, even after controlling for age, gender, charge severity, prior criminal history, and actual two-year recidivism.

2. **Age effect:** Defendants under 25 are roughly 2.5 times more likely to receive a high score compared to those aged 25-45.

3. **Gender effect:** Female defendants are about 19% more likely than male defendants to receive a higher score.

4. **False positive rates:** African-American defendants have a substantially higher FPR (\u2248 0.37) compared to Caucasian defendants (\u2248 0.10), meaning Black defendants who did *not* recidivate were much more likely to be incorrectly classified as high risk.

5. **False negative rates:** Caucasian defendants have a higher FNR (\u2248 0.54) compared to African-American defendants (\u2248 0.17), meaning white defendants who *did* recidivate were more likely to be missed by the model.

6. **Disparity table:** The `delta_FPR` and `delta_FNR` columns quantify these gaps relative to the Caucasian group. African-Americans show a positive `delta_FPR` (\u2248 +0.26) and a negative `delta_FNR` (\u2248 -0.36), indicating the model is systematically biased against them.

### Note on Python vs. R Differences

Minor differences in coefficient values and derived metrics (on the order of 0.001 or less) may occur because Python's `statsmodels` and R's `glm()` use slightly different numerical optimization defaults. The substantive conclusions are identical.